# Project: Budgeted Interactive Movie Recommender Using RL

## 1. Presentation Overview & Speaker Notes
This section contains the core narrative for the project proposal.

### Slide 1: Introduction
* **Goal:** A smarter movie recommender that learns when to ask and when to recommend.
* **The Big Idea:** Better recommendations need better information.
* **The Challenge:** Too many questions can annoy the user (User Friction).
* **Core Takeaway:** The system learns when asking is worth the cost.

### Slide 2: Why This Project Matters
* **Status Quo:** Many systems try to act immediately, leading to weak recommendations.
* **Main Trade-Off:** More information improves recommendations, but too much interaction creates friction.
* **Solution:** A system that gathers "just enough" information.

### Slide 3: How it Works (The RL Loop)
* **Workflow:** User Starts Session $\rightarrow$ System Estimates Knowledge $\rightarrow$ Agent decides "Ask" or "Recommend" $\rightarrow$ User Responds $\rightarrow$ System Learns.
* **Mechanism:** The system improves through repeated interaction and feedback over time.

## 2. The Reinforcement Learning Framework
This project treats recommendation as a sequential decision-making process.

* **Agent:** The recommendation system.
* **Environment:** The user profile/session.
* **State ($s$):** The current user information gathered and the remaining question budget.
* **Actions ($a$):** Discrete choice between "Ask a Question" or "Recommend a Movie".
* **Reward ($r$):** Positive for a correct recommendation; negative (penalty) for questions.

### A. The Agent
The **Agent** is the core decision-maker of the recommendation system. Its goal is to learn a strategic policy that balances the need for information against the risk of user friction. It must decide at each step whether the expected increase in recommendation quality justifies the "cost" of asking the user another question.

### B. The Environment
The **Environment** consists of the user's hidden preferences and the movie database. It acts as a black box that responds to the Agent's actions. For this project, we use a custom-built simulator to model different user profiles and their patience levels.

### C. The State ($s$)
The **State** is the Agent's mathematical representation of the current session. It includes:
* **Gathered Information**: A vector of movie attributes the user has already confirmed.
* **Confidence/Uncertainty**: A measure of how much the Agent knows vs. what it needs to know for a solid recommendation.
* **Budget Status**: How many questions remain before the session hits the friction limit.

### D. The Actions ($a$)
The Agent has a discrete action space:
1. **ASK**: Select a question to reduce uncertainty (e.g., asking about a specific genre).
2. **RECOMMEND**: Present a final movie suggestion based on the current state.

### E. The Reward ($r$)
The **Reward** function is designed to incentivize efficiency and accuracy:

$$r = (R_{success} \times \text{is\ accepted}) - (C_{friction} \times \text{num\ questions})$$

* **$R_{success}$**: A high positive value rewarded only if the user accepts the recommendation.
* **$C_{friction}$**: A negative penalty applied for every question asked to ensure the system respects the user's time.

## 3. Mathematical Foundations

### A. Q-Learning (The Baseline)
Used for smaller, discrete environments. It utilizes the Bellman Equation to update values:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

* **$\alpha$**: Learning rate.
* **$\gamma$**: Discount factor for future rewards.

### B. Deep Q-Network (DQN)
Extends Q-Learning using a Neural Network to handle complex state spaces. The network minimizes MSE loss:

$$L(\theta) = \mathbb{E} \left[ \left( r + \gamma \max_{a'} Q(s', a'; \theta^{-}) - Q(s, a; \theta) \right)^2 \right]$$

* **$\theta$**: Primary network weights.
* **$\theta^{-}$**: Target network weights for stability.

### C. Proximal Policy Optimization (PPO)
Advanced policy-based method used to test stability. Uses a clipped surrogate objective:

$$L^{CLIP}(\theta) = \hat{\mathbb{E}}_t \left[ \min \left( r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

* **$r_t(\theta)$**: Probability ratio between new and old policies.
* **$\hat{A}_t$**: The advantage function.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# 1. Custom Environment Skeleton
class MovieRecEnv:
    def __init__(self, question_budget=5):
        self.budget = question_budget
        self.state = None 
        
    def reset(self):
        self.budget = 5
        return self.state

    def step(self, action):
        # Action 0: Ask, Action 1: Recommend
        if action == 0:
            self.budget -= 1
            reward = -1 # Friction cost
            done = self.budget <= 0
        else:
            reward = 10 # High reward for success
            done = True
        return self.state, reward, done

# 2. DQN Architecture Skeleton
class DQNAgent(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQNAgent, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )
        
    def forward(self, x):
        return self.fc(x)

## 4. Project Summary
* **Problem:** Recommenders perform poorly without enough user data.
* **Idea:** Use RL to decide whether to ask a question or recommend immediately.
* **Constraint:** Limited question budget to minimize user friction.
* **Outcome:** A smarter interactive recommender balancing information and experience.